In [2]:
%pip install scikit-learn


   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.3 MB 5.1 MB/s eta 0:00:02
   ------- -------------------------------- 1.6/8.3 MB 4.7 MB/s eta 0:00:02
   ------------- -------------------------- 2.9/8.3 MB 4.9 MB/s eta 0:00:02
   ------------------ --------------------- 3.9/8.3 MB 5.1 MB/s eta 0:00:01
   ------------------------ --------------- 5.0/8.3 MB 5.3 MB/s eta 0:00:01
   ------------------------------ --------- 6.3/8.3 MB 5.3 MB/s eta 0:00:01
   ----------------------------------- ---- 7.3/8.3 MB 5.3 MB/s eta 0:00:01
   ---------------------------------------  8.1/8.3 MB 5.4 MB/s eta 0:00:01
   ---------------------------------------- 8.3/8.3 MB 4.6 MB/s  0:00:01

   ---------------------------------------- 0/3 [threadpoolctl]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   -----------

1. Importación de Librerías y Carga de Datos
En este paso cargamos el dataset procesado que generamos en la Notebook 02. Es obligatorio trabajar sobre los datos limpios para evitar que el ruido distorsione los componentes

In [7]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Carga del dataset procesado (asegurarse de que la ruta sea correcta)
df = pd.read_csv('../data/processed/dataset_final.csv')

print(f"Dataset cargado con {df.shape[0]} filas y {df.shape[1]} columnas.")

Dataset cargado con 7662 filas y 8 columnas.


El escalamiento con StandardScaler es una condición no negociable
. Dado que variables como monthly_watch_time_mins tienen magnitudes mucho mayores que age o customer_support_tickets, sin el escalado, la variable con el rango numérico más grande dominaría artificialmente el cálculo de la varianza
. Al estandarizar, llevamos todas las variables a una media de 0 y un desvío estándar de 1, permitiendo que todas contribuyan equitativamente al análisis

 Selección de Variables y Estandarización
PCA solo trabaja con variables cuantitativas
. Seleccionamos las columnas numéricas clave detectadas en la inspección inicial

In [8]:
# 1. Selección de variables numéricas
columnas_num = ['age', 'monthly_watch_time_mins', 'customer_support_tickets']
df_num = df[columnas_num]

# 2. Aplicación de StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_num)

# Verificación rápida de la media (~0) y desvío (~1)
print("Estadísticas tras el escalado (Media | Desvío):")
print(X_scaled.mean(axis=0).round(2), X_scaled.std(axis=0).round(2))

Estadísticas tras el escalado (Media | Desvío):
[ 0.  0. -0.] [1. 1. 1.]


Ejecución de PCA y Análisis de Varianza Explicada
El objetivo es determinar cuántos componentes son suficientes para representar la mayor parte de la variabilidad sin pérdida crítica de información
. La cátedra sugiere retener al menos el 80% de la varianza total

In [9]:
# 1. Inicializar PCA sobre todos los componentes posibles
pca = PCA()
pca.fit(X_scaled)

# 2. Calcular varianza explicada y acumulada
varianza_expl = pca.explained_variance_ratio_
varianza_acum = np.cumsum(varianza_expl)

# 3. Visualización: Gráfico de Codo (Scree Plot)
fig_pca = px.area(
    x=range(1, len(varianza_acum) + 1),
    y=varianza_acum,
    title='Varianza Explicada Acumulada por Componentes Principales',
    labels={'x': 'Número de Componentes', 'y': 'Varianza Acumulada'},
    markers=True,
    template="plotly_white"
)

# Agregamos una línea de referencia en el 80%
fig_pca.add_hline(y=0.8, line_dash="dash", line_color="red", annotation_text="Umbral 80%")
fig_pca.show()

# Imprimir reporte de varianza
for i, var in enumerate(varianza_acum):
    print(f"PC{i+1}: captura el {var*100:.2f}% de la varianza total.")

PC1: captura el 33.66% de la varianza total.
PC2: captura el 67.04% de la varianza total.
PC3: captura el 100.00% de la varianza total.


Análisis de Loadings (Cargas)
Los componentes principales no tienen nombre; el analista debe interpretarlos mirando qué variables originales "pesan" más en cada uno

In [10]:
# Creamos un DataFrame con los loadings (pesos) de cada variable en los componentes
loadings = pd.DataFrame(
    pca.components_.T,
    index=columnas_num,
    columns=[f'PC{i+1}' for i in range(len(columnas_num))]
)

print("Matriz de Loadings (Cargas):")
display(loadings.round(3))

Matriz de Loadings (Cargas):


,PC1,PC2,PC3
age,0.083,0.934,-0.348
monthly_watch_time_mins,-0.692,0.306,0.654
customer_support_tickets,0.717,0.187,0.672


Interpretación de los Resultados (Markdown)
Debés redactar un párrafo final explicando qué encontraste en tu dataset específico. Aquí tienes un ejemplo base:
Interpretación de PCA:
Varianza: Se observa que con los primeros X componentes (ajustar según tu gráfico) logramos superar el umbral del 80% de la varianza acumulada
. Esto evidencia que existe redundancia informativa entre las variables originales y que podemos reducir la complejidad del dataset de 3 a 2 dimensiones con una pérdida de información mínima
.
Loadings: Al analizar el PC1, se nota una fuerte influencia de la variable monthly_watch_time_mins (valor cercano a 1 o -1), lo que sugiere que este componente captura principalmente la intensidad de uso del servicio
. Por otro lado, el PC2 está dominado por customer_support_tickets, capturando una dimensión de fricción o problemas técnicos del usuario
.
Decisión: Estos nuevos componentes podrán ser utilizados en etapas futuras para segmentar usuarios de manera más eficiente, eliminando el sesgo de las escalas originales